## Start with imports

In [8]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import label_binarize

from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

## Setup the device for PyTorch.
Some group members have Apple Silicon Macs which are quite powerful, so we add MPS as a device option as well.

In [9]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
)

print(f"The device that will be used is {device}")

The device that will be used is cuda


## Setup Random Seeds

In [10]:
torch.manual_seed(67)
np.random.seed(67)

## Download Dataset for Google Colab Purposes

In [11]:
!gdown 1-D-EU9vYdyveomxLwq3XnIXLEtl9YO4s -O data.parquet

Downloading...
From: https://drive.google.com/uc?id=1-D-EU9vYdyveomxLwq3XnIXLEtl9YO4s
To: /content/data.parquet
100% 44.8M/44.8M [00:00<00:00, 57.0MB/s]


## Load Data from Parquet
Couple of things that happen here:
- Make sure that the data is sorted so each game is ordered by time
- Drop the last row of each game, since the change in price is NaN

In [12]:
df = pd.read_parquet('data.parquet')
df = df.sort_values(['game_id', 'timestamp']).reset_index(drop=True)
df = df.dropna(subset=['delta']).reset_index(drop=True)

## Split up the Data
Next, we need to split up the data (similar to past programming assignments).

In [13]:
games = df['game_id'].unique().copy()
np.random.shuffle(games)

train_games = set(games[:int(0.70 * len(games))])
val_games = set(games[int(0.70 * len(games)):int(0.85 * len(games))])
test_games = set(games[int(0.85 * len(games)):])

split_col = []
for game_id in df['game_id'].values:
    if game_id in train_games:
        split_col.append('train')
    elif game_id in val_games:
        split_col.append('val')
    else:
        split_col.append('test')

split_arr = np.array(split_col)
training_index = np.where(split_arr == 'train')[0]
validation_index = np.where(split_arr == 'val')[0]
test_index = np.where(split_arr == 'test')[0]


## Standardize Numerical Features
Adjust all the numeric columns to have a mean of 0 and a standard deviation of 1.

In [14]:
list_of_numerical_features = ['price_close', 'price_open', 'price_high', 'price_low', 'price_mean', 'volume', 'comment_count', 'avg_score']
raw_numerical_values = df[list_of_numerical_features].fillna(0).astype(np.float32).values

mean = raw_numerical_values[training_index].mean(axis=0)
standard_deviation = raw_numerical_values[training_index].std(axis=0) + 1e-7
numeric = ((raw_numerical_values - mean) / standard_deviation).astype(np.float32)

## Text Embeddings
Each comment will be embedded using 2 frozen encoders:
- MiniLM: this is used to produce strong semantic similarity embeddings.
- Twitter-RoBERTa: this was trained specifically on Tweets. Posts on Twitter closely
follow the style of comments on Reddit, so this should be a good fit.

After using these encoders, we get a 384-dimensional vector from MiniLM and a 768-dimensional vector from Twitter-RoBERTa. These vectors are then concatenated together.

In [15]:
embedding_cache = "comment_embeddings.npz"

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1e-6)

def roberta_embedding(texts, tokenizer, model, batch_size=64):
    vecs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='Twitter-RoBERTa', leave=False):
            batch = texts[i:i+batch_size]
            tokens = tokenizer(batch, padding=True, truncation=True, max_length=96, return_tensors='pt').to(device)
            hidden = model(**tokens).last_hidden_state
            vecs.append(mean_pool(hidden, tokens['attention_mask']).cpu().numpy())

    if vecs:
        return np.vstack(vecs)
    else:
        return np.zeros((0, model.config.hidden_size), dtype=np.float32)

def compute_embeddings(df, path_to_embeddings_cache):
    if os.path.exists(path_to_embeddings_cache):
        z = np.load(path_to_embeddings_cache)
        if len(z['emb']) == len(df):
            return z['emb'].astype(np.float32)

    print("Now loading encoders!!!!")

    mini_lm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=str(device))
    twitter_roberta_tokenizer = AutoTokenizer.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest')
    twitter_roberta_model = AutoModel.from_pretrained('cardiffnlp/twitter-roberta-base-sentiment-latest').to(device).eval()
    embedding_dimension = mini_lm.get_sentence_embedding_dimension() + twitter_roberta_model.config.hidden_size

    flattened_text = []
    flattened_row_ids = []
    for row_index in range(len(df)):
        comments = df['comments_list'].iloc[row_index]
        if comments is not None:
            for comment in comments:
                if isinstance(comment, str) and comment.strip():
                    flattened_text.append(comment[:500])
                    flattened_row_ids.append(row_index)

    minilm_embeddings = mini_lm.encode(flattened_text, batch_size=128, show_progress_bar=True, convert_to_numpy=True)
    roberta_embeddings = roberta_embedding(flattened_text, twitter_roberta_tokenizer, twitter_roberta_model)
    comment_embeddings = np.concatenate([minilm_embeddings, roberta_embeddings], axis=1)

    sum_of_rows = np.zeros((len(df), embedding_dimension), dtype=np.float32)
    count_of_rows = np.zeros(len(df), dtype=np.int32)

    for embedding, row_index in zip(comment_embeddings, flattened_row_ids):
        sum_of_rows[row_index] += embedding
        count_of_rows[row_index] += 1

    row_embeddings = np.zeros((len(df), embedding_dimension), dtype=np.float32)
    for i in range(len(df)):
        if count_of_rows[i] > 0:
            row_embeddings[i] = sum_of_rows[i] / count_of_rows[i]

    np.savez(path_to_embeddings_cache, emb=row_embeddings)
    return row_embeddings


## Call the functions from the previous cell

In [16]:
text_embedding = compute_embeddings(df, embedding_cache)
print("Done w/Embedding!")

Now loading encoders!!!!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

... (16 outputs left out)

Batches:   0%|          | 0/4363 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Twitter-RoBERTa:   0%|          | 0/8726 [00:00<?, ?it/s]

Done w/Embedding!


## Build the PyTorch Dataset Class

In [17]:
W = 5

direction_to_class_index = {
    'DOWN': 0,
    'FLAT': 1,
    'UP': 2
}

class_label_indices = df['label'].map(direction_to_class_index).astype(np.int64).values
game_ids = df['game_id'].values

game_first_row = {}
for i in range(len(df)):
    if game_ids[i] not in game_first_row:
        game_first_row[game_ids[i]] = i

class ModelDataset(Dataset):
    def __init__(self, indices):
        self.indices = [int(i) for i in indices if int(i) - game_first_row[game_ids[int(i)]] + 1 >= W]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, j):
        i = self.indices[j]
        rows = np.arange(i - W + 1, i + 1)

        return {
            'text': torch.from_numpy(text_embedding[rows]),
            'num': torch.from_numpy(numeric[rows]),
            'label': int(class_label_indices[i])
        }


## Build the Main Model

In [ ]:
class MainModel(nn.Module):
    def __init__(self, text_dim, num_dim, d_model=256, n_heads=4, ffn_dim=512, dropout=0.25):
        super().__init__()

        self.text_projection = nn.Sequential(
            nn.LayerNorm(text_dim),
            nn.Linear(text_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.no_comments = nn.Parameter(torch.randn(d_model) * 0.02)

        self.num_mlp = nn.Sequential(
            nn.LayerNorm(num_dim),
            nn.Linear(num_dim, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model)
        )

        self.gate = nn.Linear(2 * d_model, d_model)
        self.fuse_t = nn.Linear(d_model, d_model)
        self.fuse_n = nn.Linear(d_model, d_model)

        self.positional_embedding = nn.Embedding(W, d_model)
        self.classifier_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=ffn_dim,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=1)
        self.norm = nn.LayerNorm(d_model)

        self.classification_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 3)
        )

    def encode_minute(self, text, num):
        has_comments = text.abs().sum(dim=-1) > 1e-6

        text_vector = self.text_projection(text)
        no_comments = self.no_comments.unsqueeze(0).expand_as(text_vector)
        text_vector = torch.where(has_comments.unsqueeze(-1), text_vector, no_comments)

        num_vec = self.num_mlp(num)

        gate = torch.sigmoid(self.gate(torch.cat([text_vector, num_vec], dim=-1)))
        fused = gate * torch.tanh(self.fuse_t(text_vector)) + (1 - gate) * torch.tanh(self.fuse_n(num_vec))
        return fused

    def forward(self, batch):
        text = batch['text']
        num = batch['num']
        B = text.shape[0]

        tokens = [self.encode_minute(text[:, t, :], num[:, t, :]) for t in range(W)]
        x = torch.stack(tokens, dim=1)

        positions = torch.arange(W, device=x.device).unsqueeze(0).expand(B, -1)
        x += self.positional_embedding(positions)

        cls = self.classifier_token.expand(B, 1, -1)
        x = torch.cat([cls, x], dim=1)

        h = self.transformer(x)
        cls_out = self.norm(h[:, 0, :])

        return self.classification_head(cls_out)


## Model Training

In [ ]:
def compute_class_weights(targets):
    counts = np.bincount(targets, minlength=3).astype(np.float32)
    weights = counts.sum() / (3 * counts)
    return np.minimum(weights, 2.0)

def evaluate_model(model, loader):
    model.eval()
    all_labels = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for batch in loader:
            for k in ('text', 'num'):
                batch[k] = batch[k].to(device)

            logits = model(batch)
            probabilities = F.softmax(logits, dim=-1).cpu().numpy()

            all_labels.extend(batch['label'].numpy().tolist())
            all_predictions.extend(probabilities.argmax(axis=-1).tolist())
            all_probabilities.append(probabilities)

    y = np.array(all_labels)
    y_hat = np.array(all_predictions)
    probabilities = np.concatenate(all_probabilities, axis=0)
    y_onehot = label_binarize(y, classes=[0, 1, 2])
    brier = float(np.mean(np.sum((probabilities - y_onehot) ** 2, axis=1)))

    return {
        'accuracy': float(accuracy_score(y, y_hat)),
        'macro_f1': float(f1_score(y, y_hat, average='macro')),
        'brier': brier,
        'y_actual': y,
        'y_predicted': y_hat,
        'probabilities': probabilities
    }

def train_model(epochs=50, batch_size=32, lr=3e-4, dropout=0.25):
    train_loader = DataLoader(ModelDataset(training_index), batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(ModelDataset(validation_index), batch_size=256, shuffle=False, num_workers=0)

    model = MainModel(text_embedding.shape[1], numeric.shape[1], dropout=dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    num_steps = epochs * len(train_loader)
    warmup = max(1, int(0.05 * num_steps))
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[
            torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1e-3, total_iters=warmup),
            torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_steps - warmup)
        ],
        milestones=[warmup]
    )

    classifier_weights = torch.tensor(compute_class_weights(class_label_indices[training_index]), device=device)

    best_f1 = -1.0
    best_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        num_batches = 0

        for batch in train_loader:
            for k in ('text', 'num'):
                batch[k] = batch[k].to(device, non_blocking=True)
            y = batch['label'].to(device)

            logits = model(batch)
            loss = F.cross_entropy(logits, y, weight=classifier_weights, label_smoothing=0.1)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            total_loss += loss.item()
            num_batches += 1

        val = evaluate_model(model, val_loader)
        print(f'Epoch {epoch+1:02d}/{epochs}  loss={total_loss/num_batches:.4f}  '
              f'val_acc={val["accuracy"]:.4f}  val_f1={val["macro_f1"]:.4f}')

        if val['macro_f1'] > best_f1:
            best_f1 = val['macro_f1']
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model


## Train the model

In [26]:
model = train_model()

/tmp/ipykernel_689/3057684669.py:37: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer=encoder_layer, num_layers=1)


Epoch 01/50  loss=1.0786  val_acc=0.4592  val_f1=0.4431
Epoch 02/50  loss=1.0387  val_acc=0.4475  val_f1=0.4425
Epoch 03/50  loss=1.0251  val_acc=0.5235  val_f1=0.4842
Epoch 04/50  loss=1.0165  val_acc=0.4992  val_f1=0.4083
Epoch 05/50  loss=1.0123  val_acc=0.5271  val_f1=0.4545
Epoch 06/50  loss=1.0083  val_acc=0.5307  val_f1=0.4053
Epoch 07/50  loss=1.0063  val_acc=0.5250  val_f1=0.4034
Epoch 08/50  loss=1.0035  val_acc=0.4986  val_f1=0.4740
Epoch 09/50  loss=1.0013  val_acc=0.5245  val_f1=0.4243
Epoch 10/50  loss=0.9989  val_acc=0.5163  val_f1=0.4910
Epoch 11/50  loss=0.9967  val_acc=0.5103  val_f1=0.4164
Epoch 12/50  loss=0.9945  val_acc=0.5005  val_f1=0.4158
Epoch 13/50  loss=0.9904  val_acc=0.5307  val_f1=0.4640
Epoch 14/50  loss=0.9882  val_acc=0.5321  val_f1=0.5003
Epoch 15/50  loss=0.9851  val_acc=0.5223  val_f1=0.4980
Epoch 16/50  loss=0.9824  val_acc=0.5245  val_f1=0.4241
Epoch 17/50  loss=0.9795  val_acc=0.5236  val_f1=0.4318
Epoch 18/50  loss=0.9775  val_acc=0.5305  val_f1

## Evaluate on Test Set

In [ ]:
test_loader = DataLoader(ModelDataset(test_index), batch_size=256, shuffle=False, num_workers=0)
results = evaluate_model(model, test_loader)

print(f'Accuracy: {results["accuracy"]:.4f}')
print(f'Macro F1: {results["macro_f1"]:.4f}')
print(f'Brier:    {results["brier"]:.4f}')
print(classification_report(results['y_actual'], results['y_predicted'], target_names=['DOWN', 'FLAT', 'UP'], digits=3))
print('Baseline LR: acc=0.473  macro F1=0.445  Brier=0.615')


Accuracy: 0.5437
Macro F1: 0.5183
Brier:    0.5549
              precision    recall  f1-score   support

        DOWN      0.397     0.416     0.406      2206
        FLAT      0.757     0.643     0.696      3730
          UP      0.413     0.502     0.453      2123

    accuracy                          0.544      8059
   macro avg      0.522     0.520     0.518      8059
weighted avg      0.568     0.544     0.553      8059

Baseline LR: acc=0.473  macro F1=0.445  Brier=0.615


In [37]:
names_of_classes = ['DOWN', 'FLAT', 'UP']
test_rows = ModelDataset(test_index).indices

# grab the results
probabilities = results['probabilities']
y_actual = results['y_actual']
y_predicted = results['y_predicted']

confidence = probabilities.max(axis=1)
correct = (y_actual == y_predicted)

def has_valid_comments(comments):
    if comments is None:
        return False
    return any(isinstance(c, str) and c.strip() for c in comments)

has_text = np.array([has_valid_comments(df['comments_list'].iloc[idx]) for idx in test_rows])

valid_correct = np.where(correct & has_text)[0]
valid_wrong = np.where(~correct & has_text)[0]

sorted_correct = valid_correct[np.argsort(-confidence[valid_correct])]
sorted_wrong = valid_wrong[np.argsort(-confidence[valid_wrong])]

# N = num of examples to show
N = 5
top_correct = sorted_correct[:N]
top_wrong = sorted_wrong[:N]

def show_example_from_model(title, pos):
    df_row = test_rows[pos]
    predicted_class = names_of_classes[y_predicted[pos]]
    actual_class = names_of_classes[y_actual[pos]]
    c = confidence[pos]

    print(f"=== {title} ===")
    print(f'Predicted Class: {predicted_class} ({c:.1%} confidence)')
    print(f'Actual Class: {actual_class}')
    print(f'Delta: {df["delta"].iloc[df_row]:+.4f}\n')

    for offset, row_i in enumerate(range(df_row - W + 1, df_row + 1)):
        row = df.iloc[row_i]
        comments = row['comments_list']
        label = 'NOW' if offset == W - 1 else f't-{W-1-offset}'
        print(f'  [{label}] price={row["price_close"]:.2f}  {comments}')
        print()

print(f"top {N} CORRECT preds")
for i, pos in enumerate(top_correct):
    show_example_from_model(f'correct pred #{i+1}', pos)

print(f"top {N} incorrect preds")
for i, pos in enumerate(top_wrong):
    show_example_from_model(f'incorrect pred #{i+1}', pos)


top 5 CORRECT preds
=== correct pred #1 ===
Predicted Class: UP (95.9% confidence)
Actual Class: UP
Delta: +0.0100

  [t-4] price=0.40  []

  [t-3] price=0.40  []

  [t-2] price=0.41  []

  [t-1] price=0.41  []

  [NOW] price=0.39  ["I can't wait to watch this game ngl"
 'I can’t wait for this to impact the rest of my week']

=== correct pred #2 ===
Predicted Class: UP (95.1% confidence)
Actual Class: UP
Delta: +0.0200

  [t-4] price=0.45  ['We really know how to waste careers man....'
 'Bill is averaging 9 YPC. Why does he only have five carries?']

  [t-3] price=0.45  ["Really was a highlight return. Glad it didn't count, but feel bad for the returner"
 "Washington is doing its part with Terry McLaurin's career."]

  [t-2] price=0.44  ['Lol...they made a season 2. My wife and I tried watching. It was awful.'
 "Did you know you can't touch the kickers leg unless you touch the ball? Thanks for a free 15, 3 points, and the ball for the 2nd half"]

  [t-1] price=0.44  ['We Completely cha